# Deploy Registry Assets with a System-Assigned Identity

Deploy both the model and environment from the configured external Azure ML registry to a new private, Microsoft Entra-authenticated managed online endpoint. The endpoint receives its own system-assigned identity; no user-assigned identity is read or attached.

Complete [08 Registry sharing](08_register_assets_to_registry.ipynb) first. The default assets are the foundation JSON model and inference environment. The notebook caller needs registry asset access, such as **AzureML Registry User**, and endpoint-deployment permissions. The caller and endpoint also need the required private connectivity and artifact access to registry Storage/ACR. Changing credentials does not create that connectivity.

Run the cells individually. Preparation is read-only apart from generated local files. Endpoint creation and deployment are separate Azure mutations; deployment can incur charges. No existing endpoint is deleted, no RBAC role is explicitly granted, and no firewall is opened.

**Sources:** Adapted from [06 Managed endpoint](06_deploy_online_endpoint.ipynb), including its MIT-licensed Azure ML example attribution, and the [Azure ML registry sharing guide](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-share-models-pipelines-across-workspaces-with-registries?view=azureml-api-2).

In [ ]:
from pathlib import Path
import ast
import json
import textwrap
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    CodeConfiguration, IdentityConfiguration,
    ManagedOnlineDeployment, ManagedOnlineEndpoint,
)
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import AzureCliCredential, ManagedIdentityCredential
from dotenv import dotenv_values

notebook_path = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "09_deploy_registry_online_endpoint.ipynb"))
SUITE_ROOT = notebook_path.resolve().parents[1]
env = dotenv_values(SUITE_ROOT / ".env", encoding="utf-8-sig", interpolate=False)
SUBSCRIPTION_ID = (env.get("AZURE_SUBSCRIPTION_ID") or "").strip()
RESOURCE_GROUP = (env.get("AZURE_RESOURCE_GROUP") or "").strip()
WORKSPACE_NAME = (env.get("AZUREML_WORKSPACE_NAME") or "").strip()
REGISTRY_NAME = (env.get("AZUREML_REGISTRY_NAME") or "").strip()
if not all((SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError(f"Fill workspace and registry coordinates in {SUITE_ROOT / '.env'}.")
use_managed_identity = (env.get("USE_MANAGED_IDENTITY", "true") or "").strip().lower()
if use_managed_identity not in {"true", "false"}:
    raise ValueError("USE_MANAGED_IDENTITY must be true or false in .env.")
credential = ManagedIdentityCredential(process_timeout=60) if use_managed_identity == "true" else AzureCliCredential(process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
print(f"Workspace: {WORKSPACE_NAME}; external registry: {REGISTRY_NAME}")

## Select Both Assets from the Registry

Change the asset names here if needed. Each latest label is resolved once, then its immutable version is pinned in the deployment URI. Neither asset falls back to the workspace. The endpoint-name prefix is limited to 17 characters to fit Azure's 32-character limit; change the name if that prefix is not unique.

In [ ]:
ENDPOINT_NAME = f"{WORKSPACE_NAME[:14]}-registry-sami-v2"
DEPLOYMENT_NAME = "registry"
MODEL_NAME = "readiness-taxi-model"
ENVIRONMENT_NAME = "readiness-taxi-environment"
INSTANCE_TYPE = "Standard_DS3_v2"
registry_client = MLClient(credential=credential, registry_name=REGISTRY_NAME)
registry_model = registry_client.models.get(MODEL_NAME, label="latest")
registry_environment = registry_client.environments.get(ENVIRONMENT_NAME, label="latest")
MODEL_URI = f"azureml://registries/{REGISTRY_NAME}/models/{MODEL_NAME}/versions/{registry_model.version}"
ENVIRONMENT_URI = f"azureml://registries/{REGISTRY_NAME}/environments/{ENVIRONMENT_NAME}/versions/{registry_environment.version}"
print(f"Model: {MODEL_URI}")
print(f"Environment: {ENVIRONMENT_URI}")

## Prepare the Scorer and Deployment

The scorer matches the foundation model's `model.json` contract. This cell only generates local payload files and constructs definitions. It does not create an endpoint or deployment.

In [ ]:
generated_dir = SUITE_ROOT / "outputs/generated/foundations/registry_online"
generated_code_dir = generated_dir / "code"
generated_code_dir.mkdir(parents=True, exist_ok=True)
(generated_code_dir / ".amlignore").write_text("__pycache__/\n*.py[cod]\n", encoding="utf-8")
score_path = generated_code_dir / "score.py"
request_path = generated_dir / "request.json"
score_source = r'''
import json
import os
from pathlib import Path
import pandas as pd

_model = None

def init():
    global _model
    matches = list(Path(os.environ["AZUREML_MODEL_DIR"]).rglob("model.json"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one model.json, found {len(matches)}")
    _model = json.loads(matches[0].read_text(encoding="utf-8"))

def run(raw_data):
    payload = json.loads(raw_data) if isinstance(raw_data, (str, bytes)) else raw_data
    input_data = payload.get("input_data", {})
    columns = input_data.get("columns")
    rows = input_data.get("data")
    if columns != _model["features"]:
        raise ValueError(f"Expected columns in this order: {_model['features']}")
    if not isinstance(rows, list) or not rows:
        raise ValueError("Request must contain at least one row")
    frame = pd.DataFrame(rows, columns=columns).apply(pd.to_numeric, errors="raise")
    predictions = _model["intercept"]
    for feature, coefficient in zip(_model["features"], _model["coefficients"], strict=True):
        predictions = predictions + frame[feature] * coefficient
    return {"predictions": predictions.astype(float).tolist()}
'''
score_source = textwrap.dedent(score_source).lstrip()
ast.parse(score_source, filename=str(score_path))
score_path.write_text(score_source, encoding="utf-8")
request = {"input_data": {"columns": ["tripDistance", "passengerCount"], "data": [[2.5, 1], [7.0, 2]]}}
request_path.write_text(json.dumps(request, indent=2) + "\n", encoding="utf-8")
endpoint_definition = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="Registry-backed readiness endpoint with a system-assigned identity",
    auth_mode="aad_token",
    identity=IdentityConfiguration(type="system_assigned"),
    public_network_access="disabled",
    tags={"validation": "workspace-readiness", "purpose": "registry-inference"},
)
deployment_definition = ManagedOnlineDeployment(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
    model=MODEL_URI,
    environment=ENVIRONMENT_URI,
    code_configuration=CodeConfiguration(code=str(generated_code_dir), scoring_script="score.py"),
    instance_type=INSTANCE_TYPE,
    instance_count=1,
)
print(f"Prepared {ENDPOINT_NAME}/{DEPLOYMENT_NAME}; no deployment submitted.")

## Create the System-Assigned Endpoint

This creates or reuses only the new registry endpoint. Review its principal ID and ensure the required registry-backed Storage/ACR access is in place before deploying. An existing user-assigned endpoint is rejected rather than modified. Azure authorization failures are not bypassed.

In [ ]:
try:
    endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
except ResourceNotFoundError:
    endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint_definition).result()
identity_type = endpoint.identity.type if endpoint.identity else None
if not identity_type or identity_type.replace("_", "").lower() != "systemassigned":
    raise ValueError(f"Endpoint {endpoint.name} has identity type {identity_type!r}; expected 'system_assigned'. Choose a new ENDPOINT_NAME.")
print(f"Endpoint: {endpoint.name}; system-assigned principal: {endpoint.identity.principal_id}")

## Deploy, Invoke, and Promote

This creates a billable deployment using both pinned registry URIs. It tests the deployment directly before promoting it to 100% of this new endpoint's traffic. Private invocation requires routing and DNS from the notebook host.

In [ ]:
deployment = ml_client.online_deployments.begin_create_or_update(deployment_definition).result()
if deployment.provisioning_state != "Succeeded":
    raise RuntimeError(f"Deployment state is {deployment.provisioning_state}")
raw_response = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name=DEPLOYMENT_NAME,
    request_file=str(request_path),
)
response = json.loads(raw_response)
if response.get("predictions") != [9.2, 20.2]:
    raise RuntimeError(f"Unexpected scoring response: {response}")
print(response)
endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
endpoint.traffic = {DEPLOYMENT_NAME: 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Traffic promoted to {DEPLOYMENT_NAME}; endpoint identity is system-assigned.")

## Expected Result

Both deployment asset references point to immutable versions in the external registry. The new endpoint uses a system-assigned identity, invocation returns `[9.2, 20.2]`, and the tested deployment receives 100% of its traffic. Existing endpoints remain untouched. No model data collection is configured.